In [5]:
import os
import sys

os.path.dirname(sys.executable)


# from main_nolightning import initialize, configure
import pandas as pd
import numpy as np
from einops import rearrange
import torch
import pytorch_lightning as pl

import warnings
from sklearn.exceptions import DataConversionWarning

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DataConversionWarning)


%reload_ext autoreload
%autoreload 2
%matplotlib inline

/home/levtel/miniconda3/envs/phm/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Dataloaders

In [6]:
from typing import Optional, List, Tuple
from abc import abstractmethod
from torch.utils.data import DataLoader, RandomSampler, BatchSampler, SequentialSampler

from sklearn.preprocessing import MinMaxScaler


def check_boolean_vector(vec):
    # Find the index of the first False value
    try:
        first_false_index = np.where(vec == False)[0][0]
    except IndexError:  # If there is no False in the vector
        return True

    # Check if all values after the first False are also False
    return not vec[first_false_index:].any() and vec[:first_false_index].all()


def time_features(
    dates: pd.Series,
    main_df=None,
    use_features=["year", "month", "day", "weekday", "hour", "minute"],
    time_col_name="Datetime",
):
    if main_df is None:
        main_df = pd.DataFrame({})
    else:
        main_df = main_df.copy()
    years = dates.apply(lambda row: row.year)
    max_year = years.max()
    min_year = years.min()

    norm_f = lambda r: 2.0 * r - 1.0
    norm_f = lambda r: r

    if "year" in use_features:
        main_df["year"] = dates.apply(
            lambda row: (row.year - min_year) / max(1.0, (max_year - min_year))
        )

    if "month" in use_features:
        main_df["month"] = dates.apply(lambda row: norm_f((row.month - 1) / 11.0), 1)
    if "day" in use_features:
        main_df["day"] = dates.apply(lambda row: norm_f((row.day - 1) / 30.0), 1)

    if "weekday" in use_features:
        main_df["weekday"] = dates.apply(lambda row: norm_f(row.weekday() / 6.0), 1)
    if "hour" in use_features:
        main_df["hour"] = dates.apply(lambda row: norm_f((row.hour) / 23.0), 1)

    if "minute" in use_features:
        main_df["minute"] = dates.apply(lambda row: ((row.minute) / 59.0), 1)

    main_df[time_col_name] = dates
    return main_df

In [7]:
class IndexSequencer:
    def __init__(
        self,
        F: torch.Tensor,
        ts_in: int,
        ts_out: int,
        dt_ts_out: List[int] = None,
        step: int = 1,
    ) -> None:
        self.F = F

        if dt_ts_out is not None:
            dt_ts_out.sort()
            dt_ts_out = np.array(dt_ts_out)
            dt_max = dt_ts_out[-1]
            upper_limit = max(ts_out, dt_max + 1)
        else:
            upper_limit = ts_out

        N_samples = F.shape[0]

        i = torch.arange(0, N_samples - (ts_in + upper_limit) + 1, step=step)
        self.start_end_index = torch.stack([i, i + (ts_in + upper_limit)], dim=1)

        self.N_sequences = len(self.start_end_index)

    def __len__(self):
        return self.N_sequences

    def sequences_batch(
        self, index_subset: torch.Tensor | np.ndarray
    ) -> Tuple[torch.Tensor]:
        sel = self.start_end_index[index_subset]
        return self._create_sequences(sel)

    @abstractmethod
    def _create_sequences(self, sel: List[Tuple[int]]) -> Tuple[torch.Tensor]:
        pass


class AutoregressiveSequencer(IndexSequencer):
    def __init__(
        self,
        F: torch.Tensor,
        ts_in: int,
        ts_out: int,
        T: torch.Tensor = None,
        in_features_mask=None,
        out_features_mask=None,
        step: int = 10,
    ) -> None:
        super().__init__(F=F, ts_in=ts_in, ts_out=ts_out, step=step)
        self.ts_in = ts_in
        self.ts_out = ts_out
        self.in_features_mask = in_features_mask
        self.out_features_mask = out_features_mask
        self.T = F

    def _create_sequences(self, sel: List[Tuple[int]]) -> Tuple[torch.Tensor]:
        ts_in, ts_out = self.ts_in, self.ts_out
        F, T = self.F, self.T

        f_unfolded, t_unfolded = [], []
        for i, j in sel:
            f_unfolded.append(F[i : i + ts_in, ...])
            t_unfolded.append(T[i + ts_in : j, ...])

        f_unfolded = torch.stack(f_unfolded, dim=0)
        t_unfolded = torch.stack(t_unfolded, dim=0)

        if self.in_features_mask is not None:
            f_unfolded = f_unfolded[..., self.in_features_mask]

        if self.out_features_mask is not None:
            t_unfolded = t_unfolded[..., self.out_features_mask]

        return f_unfolded, t_unfolded


class ContextBatchDataset(torch.utils.data.Dataset):
    def __init__(
        self,
        Features: torch.Tensor,
        Context: torch.Tensor,
        ts_in: int,
        ts_out: int,
        out_features_mask: Optional[torch.Tensor] = None,
        step: int = 1,
        raw_data: Optional[pd.DataFrame] = None,
    ):
        self.raw_data = raw_data
        self.F_seq = AutoregressiveSequencer(
            Features, ts_in, ts_out, step=step, out_features_mask=out_features_mask
        )
        self.C_seq = AutoregressiveSequencer(Context, ts_in, ts_out, step=step)

        self.ts_in = ts_in

    def get_raw_data(self, batch_idx: List[int]):
        if self.raw_data is not None:
            ts_in = self.ts_in
            sel = self.F_seq.start_end_index[batch_idx]

            f_unfolded, t_unfolded = [], []
            for i, j in sel:
                i, j = i.item(), j.item()
                f_unfolded.append(self.raw_data.iloc[i : i + ts_in, ...])
                t_unfolded.append(self.raw_data.iloc[i + ts_in : j, ...])

            return f_unfolded, t_unfolded
        else:
            raise ValueError("No raw data available")

    def __len__(self):
        return len(self.F_seq)

    def __getitem__(self, batch_idx: List[int]):
        F_f, F_t = self.F_seq.sequences_batch(batch_idx)
        C_f, C_t = self.C_seq.sequences_batch(batch_idx)
        return C_f, F_f, C_t, F_t


class BuildingEnergyDataModule(pl.LightningDataModule):
    def __init__(
        self,
        data_dir: str,
        test_ratio: float = 0.5,
        val_ratio: float = 0.084,
        train_ratio: float = 1.0,
        train_scaled: Optional[None] = None,
        val_scaled: Optional[None] = None,
        test_scaled: Optional[None] = None,
        batch_size=64,
        samples_per_day=None,
        trend_decomp_rolling_window=384,
        ts_in=336,
        ts_out=48,
        step_train=1,
        step_val=48,
        step_test=48,
        shuffle=True,
        drop_last=False,
        num_workers=0,
        data_split_strategy="default",
        include_bl_forecast=False,
        datetime_features=True,
        forecast_feature_name="Site Total Energy (kWh)",
        forecast_all_features=False,
        trend_decomp="disabled",
        subsample_strategy=None,
        spatial_aggregation: Optional[str] = None,
    ):
        super().__init__()
        self.test_ratio = test_ratio
        self.val_ratio = val_ratio
        self.train_ratio = train_ratio

        if train_ratio < 1.0:
            assert data_split_strategy == "scale_training_set", (
                "Only scale_training_set strategy is supported"
            )

        self.samples_per_day = samples_per_day
        self.trend_decomp_rolling_window = trend_decomp_rolling_window
        self.subsample_strategy = subsample_strategy

        self.train_scaled = train_scaled
        self.val_scaled = val_scaled
        self.test_scaled = test_scaled
        self.batch_size = batch_size
        self.ts_in = ts_in
        self.ts_out = ts_out
        self.step_train = step_train
        self.step_val = step_val
        self.step_test = step_test
        self.forecast_feature_name = forecast_feature_name
        self.forecast_all_features = forecast_all_features
        self.data_split_strategy = data_split_strategy

        self.shuffle = shuffle
        self.drop_last = drop_last

        self.datetime_features = datetime_features

        self.num_workers = num_workers
        self.include_bl_forecast = include_bl_forecast
        self.trend_decomp = trend_decomp
        self.data_dir = data_dir
        self.spatial_aggregation = spatial_aggregation

    def load_raw_data(self, data_dir):
        data_path = data_dir
        data = pd.read_csv(data_path, sep=",")

        data["Unnamed: 0"] = pd.to_datetime(data["Unnamed: 0"])
        data.set_index("Unnamed: 0", inplace=True)

        return data

    def setup(self, stage: Optional[str] = None):
        if self.train_scaled is None:
            data = self.load_raw_data(self.data_dir)

            # use this to downsample the data
            data_samples_per_day = (
                data.index.to_series().groupby(data.index.date).count().iloc[0]
            )
            samples_per_day = data_samples_per_day

            assert self.samples_per_day is None, "Subsampling not implemented"
            assert self.subsample_strategy is None, "Not implemented"

            if not (
                self.subsample_strategy == "mean_1h" or self.subsample_strategy == None
            ):
                raise ValueError(
                    "Only mean_1h is supported, make sure the dataset is aligned with the small one"
                )

            if self.subsample_strategy == "select_1h":
                hourly = pd.Series(
                    data=pd.date_range(
                        start=data.index.min(), end=data.index.max(), freq="1h"
                    )
                )
                data = data.loc[hourly]
            if self.subsample_strategy == "mean_1h":
                data["Bahnlast"] = data["Bahnlast"].shift(-2).ffill()
                data = data.shift(1).bfill().resample("1h").mean()
                data = data.resample("1H").mean()

            self.data = {"full": data}

            """
            WARNING: comumns need to be in order:
            Bahnlast <forecast targets>, covaraites <contexts> ...
            The unintuitive part is that if there are N variables in the context
            sequence (the encoder input) and K (K < N) variables in the target sequence
            (the decoder input), then this implementation of Local Cross Attn
            assumes that the first K variables in the context correspond to the
            first K in the target. This means that if the context sequence is shape 
            (batch, length, N), then context[:, :, :K] gets you the context of the
            K target variables (target[..., i] is the same variable
            as context[..., i]). If this isn't true the model will still train but
            you will be connecting variables by cross attention in a very arbitrary
            way. Note that the built-in CSVDataset *does* account for this and always
            puts the target variables in the same order in the lowest indices of both
            sequences. ** If your target variables do not appear in the context sequence
            Local Cross Attention should almost definitely be turned off
            (--local_cross_attn none) **.
            """

            if self.spatial_aggregation is not None:
                print("Spatial aggregation")

            if self.trend_decomp == "disabled":
                gt = data[
                    [
                        "Site Total Energy (kWh)",
                    ]
                ]
                self.data = {"full": data, "gt": gt.copy()}
            else:
                raise ValueError("Unknown trend decomposition")

            if not self.forecast_all_features:
                # self.features_mask = (data.columns == "Bahnlast")
                self.bahnlast_mask = data.columns == self.forecast_feature_name
                self.bahnlast_mask_orig = data.columns == self.forecast_feature_name
            else:
                self.bahnlast_mask = np.ones(data.shape[1], dtype=bool)
                self.bahnlast_mask_orig = np.ones(data.shape[1], dtype=bool)

            assert check_boolean_vector(self.bahnlast_mask), (
                "Should be boolean vector in order 1111...000"
            )
            assert check_boolean_vector(self.bahnlast_mask_orig), (
                "Should be boolean vector in order 1111...000"
            )

            test_ratio = self.test_ratio
            val_ratio = self.val_ratio
            train_ratio = self.train_ratio

            cut_to_days = lambda x: (x // samples_per_day) * samples_per_day
            N_samples = cut_to_days(len(data))
            if self.data_split_strategy == "default":
                # if you modify this: this is very fragile to keep the DS day aligned!
                N_test = cut_to_days(round(test_ratio * N_samples))
                N_val = cut_to_days(round(val_ratio * N_samples))
                N_train = cut_to_days(N_samples - N_test - N_val)
                print(N_train)

                r_train = range(0, N_train)
                r_val = range(N_train, N_train + N_val)
                r_test = range(N_train + N_val, N_train + N_val + N_test)

            print("--" * 40)
            print("Dataset Split Summary:")
            print(
                f"Train: {len(r_train)} , Start Date: {data.index[r_train[0]]} , End Date: {data.index[r_train[-1]]}"
            )
            print(
                f"Val: {len(r_val)} , Start Date: {data.index[r_val[0]]} , End Date: {data.index[r_val[-1]]}"
            )
            print(
                f"Test: {len(r_test)} , Start Date: {data.index[r_test[0]]} , End Date: {data.index[r_test[-1]]}"
            )
            print("--" * 40)

            self.r_train, self.r_val, self.r_test = r_train, r_val, r_test
            self.train = data.iloc[r_train, :]
            self.val = data.iloc[r_val, :]
            self.test = data.iloc[r_test, :]

            scaler = MinMaxScaler()
            scaler.fit(self.train)
            self.train_scaled = scaler.transform(self.train)
            self.val_scaled = scaler.transform(self.val)
            self.test_scaled = scaler.transform(self.test)

            if self.datetime_features:
                dtf = time_features(
                    pd.Series(data.index),
                    use_features=["month", "day", "weekday", "hour"],
                )
                self.dtf_train = dtf.iloc[r_train, :]
                self.dtf_val = dtf.iloc[r_val, :]
                self.dtf_test = dtf.iloc[r_test, :]

                self.scaler_mask = np.concatenate(
                    (
                        np.ones(self.train_scaled.shape[1], dtype=bool),
                        np.zeros(4, dtype=bool),
                    )
                )

                self.bahnlast_mask = np.concatenate(
                    (self.bahnlast_mask, np.zeros(4, dtype=bool))
                )

                self.train_scaled = np.concatenate(
                    (
                        self.train_scaled,
                        self.dtf_train[["hour", "day", "weekday", "month"]].values,
                    ),
                    axis=1,
                )
                self.val_scaled = np.concatenate(
                    (
                        self.val_scaled,
                        self.dtf_val[["hour", "day", "weekday", "month"]].values,
                    ),
                    axis=1,
                )
                self.test_scaled = np.concatenate(
                    (
                        self.test_scaled,
                        self.dtf_test[["hour", "day", "weekday", "month"]].values,
                    ),
                    axis=1,
                )

            self.scaler = scaler

            def to_ds(ds, step, raw_data):
                # out_features_mask = torch.from_numpy(data.columns == self.forecast_feature_name).to(torch.bool) \
                #     if self.forecast_all_features else None

                out_features_mask = None

                ds = ContextBatchDataset(
                    Features=torch.from_numpy(ds[:, self.bahnlast_mask]).to(
                        torch.float32
                    ),
                    Context=torch.from_numpy(ds[:, ~self.bahnlast_mask]).to(
                        torch.float32
                    ),
                    out_features_mask=out_features_mask,
                    ts_in=self.ts_in,
                    ts_out=self.ts_out,
                    step=step,
                    raw_data=raw_data,
                )
                return ds

            self.datasets = dict(
                train_test=to_ds(self.train_scaled, self.step_test, self.train),
                train=to_ds(self.train_scaled, self.step_train, self.train),
                val=to_ds(self.val_scaled, self.step_val, self.val),
                test=to_ds(self.test_scaled, self.step_test, self.test),
            )

    def inverse_scale_bahnlast(self, X):
        x = np.zeros((X.shape[0], self.train_scaled.shape[1]))
        x[:, self.bahnlast_mask] = X

        if self.datetime_features:
            res = self.scaler.inverse_transform(x[:, self.scaler_mask])
        else:
            res = self.scaler.inverse_transform(x)

        return res[:, self.bahnlast_mask_orig]

    def _create_dataloader(self, ds, batch_size, shuffle=True):
        sampler = BatchSampler(
            RandomSampler(ds) if shuffle else SequentialSampler(ds),
            batch_size=batch_size,
            drop_last=self.drop_last,
        )
        loader_args = dict(
            sampler=sampler,
            num_workers=self.num_workers,
            # persistent_workers=True,
            batch_size=None,  # otherwise it adds a dimension with size 1 x ...
            # due to the batch sampler providing the batch dimension already
        )
        return DataLoader(ds, **loader_args)

    def train_dataloader(self):
        return self._create_dataloader(
            self.datasets["train"], batch_size=self.batch_size
        )

    def val_dataloader(self):
        return self._create_dataloader(
            self.datasets["val"], batch_size=self.batch_size, shuffle=False
        )

    def test_dataloader(self):
        return self._create_dataloader(
            self.datasets["test"], batch_size=self.batch_size, shuffle=False
        )

    def test_small_dataloader(self):
        return self._create_dataloader(
            self.datasets["test_small"], batch_size=self.batch_size, shuffle=False
        )

In [8]:
class SpatialDataModule(pl.LightningDataModule):
    def __init__(
        self,
        test_ratio: float = 0.2,
        val_ratio: float = 0.07,
        train_ratio: float = 1.0,
        train_scaled: Optional[None] = None,
        val_scaled: Optional[None] = None,
        test_scaled: Optional[None] = None,
        batch_size=64,
        samples_per_day=24,
        trend_decomp_rolling_window=384,
        ts_in=24 * 7,
        ts_out=24,
        step_train=1,
        step_val=24,
        step_test=24,
        shuffle=True,
        drop_last=False,
        num_workers=0,
        data_split_strategy="scale_training_set",
        include_bl_forecast=False,
        datetime_features=True,
        forecast_feature_name="Bahnlast",
        forecast_all_features=False,
        trend_decomp="disabled",
        subsample_strategy="mean_1h",
        spatial_aggregation: Optional[str] = None,
    ):
        super().__init__()
        self.test_ratio = test_ratio
        self.val_ratio = val_ratio
        self.train_ratio = train_ratio

        self.samples_per_day = samples_per_day
        self.trend_decomp_rolling_window = trend_decomp_rolling_window
        self.subsample_strategy = subsample_strategy

        self.train_scaled = train_scaled
        self.val_scaled = val_scaled
        self.test_scaled = test_scaled
        self.batch_size = batch_size
        self.ts_in = ts_in
        self.ts_out = ts_out
        self.step_train = step_train
        self.step_val = step_val
        self.step_test = step_test
        self.forecast_feature_name = forecast_feature_name
        self.forecast_all_features = forecast_all_features
        self.data_split_strategy = data_split_strategy

        self.shuffle = shuffle
        self.drop_last = drop_last

        self.datetime_features = datetime_features

        self.num_workers = num_workers
        self.include_bl_forecast = include_bl_forecast
        self.trend_decomp = trend_decomp
        self.spatial_aggregation = spatial_aggregation

    def setup(self, stage: Optional[str] = None):
        if self.train_scaled is None:
            data = pd.read_csv("../datasets/railway.csv", sep=",")
            data.datetime = pd.to_datetime(data.datetime)
            data.set_index("datetime", inplace=True)
            self.data = {"full": data}

            samples_per_day = self.samples_per_day

            if self.trend_decomp == "disabled":
                # What is the point to add bl and then remove it?
                bl = data["Bahnlast"]
                data.rename(columns={"Bahnlast": "Bahnlast_oldpos"}, inplace=True)
                data.insert(loc=0, column="Bahnlast", value=bl)
                data.drop(columns=["Bahnlast_oldpos"], inplace=True)
                self.data = {"full": data, "gt": bl.copy()}

            elif self.trend_decomp == "mean":
                S = self.trend_decomp_rolling_window
                self.prev_mean = data["Bahnlast"].rolling(S).mean().bfill()
                self.prev_mean_shifted = (
                    data["Bahnlast"].rolling(S).mean().shift(samples_per_day).bfill()
                )

                decomp = data["Bahnlast"] - self.prev_mean_shifted
                data.insert(loc=0, column="Bahnlast_decomp", value=decomp)

                self.data = {"full": data, "gt": data["Bahnlast"].copy()}
                data.drop(columns=["Bahnlast"], inplace=True)
            else:
                raise ValueError("Unknown trend decomposition")

            if not self.forecast_all_features:
                self.bahnlast_mask = data.columns == self.forecast_feature_name
                self.bahnlast_mask_orig = data.columns == self.forecast_feature_name
            else:
                self.bahnlast_mask = np.ones(data.shape[1], dtype=bool)
                self.bahnlast_mask_orig = np.ones(data.shape[1], dtype=bool)

            test_ratio = self.test_ratio
            val_ratio = self.val_ratio
            train_ratio = (
                self.train_ratio
            )  # why train ratio is 1? why not 1 - test - val?

            cut_to_days = lambda x: (x // samples_per_day) * samples_per_day
            N_samples = cut_to_days(len(data))
            if self.data_split_strategy == "default":
                N_test = cut_to_days(round(test_ratio * N_samples))
                N_val = cut_to_days(round(val_ratio * N_samples))
                N_train = cut_to_days(N_samples - N_test - N_val)

                r_train = range(0, N_train)
                r_val = range(N_train, N_train + N_val)
                r_test = range(N_train + N_val, N_train + N_val + N_test)
                r_test_small = range(
                    data.index.get_loc("2023-01-24 00:00:00"),
                    data.index.get_loc("2023-07-31 00:00:00"),
                )

            elif self.data_split_strategy == "scale_training_set":
                ################################################
                #    Train  |  Val  |  Test_small  | Test
                #       xxxx|xxxxxxx|xxxxxxxxxxxxxxxxxxxxx
                #   xxxxxxxx|xxxxxxx|xxxxxxxxxxxxxxxxxxxxx train
                #
                ################################################

                # we keep a relatively long test set
                r_test_small = range(
                    data.index.get_loc("2023-01-24 00:00:00"),
                    data.index.get_loc("2023-07-31 00:00:00"),
                )
                r_test = range(data.index.get_loc("2023-01-24 00:00:00"), N_samples)

                # we cut a small validation set
                N_val = cut_to_days(round(val_ratio * (N_samples - len(r_test))))
                Len_train = N_samples - N_val - len(r_test)
                N_train = cut_to_days(round(train_ratio * (Len_train)))

                r_train = range(Len_train - N_train, Len_train)
                r_val = range(Len_train, Len_train + N_val)

            elif self.data_split_strategy == "covid_training_set":
                # we keep a relatively long test set
                r_test_small = range(
                    data.index.get_loc("2023-01-24 00:00:00"),
                    data.index.get_loc("2023-07-31 00:00:00"),
                )
                r_test = range(data.index.get_loc("2023-01-24 00:00:00"), N_samples)

                # we cut a small validation set
                N_val = cut_to_days(round(val_ratio * (N_samples - len(r_test))))
                Len_train = N_samples - N_val - len(r_test)

                r_train = range(
                    data.index.get_loc("2019-12-01 00:00:00"),
                    data.index.get_loc("2021-07-31 00:00:00"),
                )
                r_val = range(Len_train, Len_train + N_val)

            self.r_train, self.r_val, self.r_test, self.r_test_small = (
                r_train,
                r_val,
                r_test,
                r_test_small,
            )
            self.train = data.iloc[r_train, :]
            self.val = data.iloc[r_val, :]
            self.test = data.iloc[r_test, :]
            self.test_small = data.iloc[r_test_small, :]

            scaler = MinMaxScaler()
            scaler.fit(self.train)
            self.train_scaled = scaler.transform(self.train)
            self.val_scaled = scaler.transform(self.val)
            self.test_scaled = scaler.transform(self.test)
            self.test_small_scaled = scaler.transform(self.test_small)

            if self.datetime_features:
                dtf = time_features(
                    pd.Series(data.index),
                    use_features=["month", "day", "weekday", "hour"],
                )
                self.dtf_train = dtf.iloc[r_train, :]
                self.dtf_val = dtf.iloc[r_val, :]
                self.dtf_test = dtf.iloc[r_test, :]
                self.dtf_test_small = dtf.iloc[r_test_small, :]

                self.scaler_mask = np.concatenate(
                    (
                        np.ones(self.train_scaled.shape[1], dtype=bool),
                        np.zeros(4, dtype=bool),
                    )
                )

                self.bahnlast_mask = np.concatenate(
                    (self.bahnlast_mask, np.zeros(4, dtype=bool))
                )

                self.train_scaled = np.concatenate(
                    (
                        self.train_scaled,
                        self.dtf_train[["hour", "day", "weekday", "month"]].values,
                    ),
                    axis=1,
                )
                self.val_scaled = np.concatenate(
                    (
                        self.val_scaled,
                        self.dtf_val[["hour", "day", "weekday", "month"]].values,
                    ),
                    axis=1,
                )
                self.test_scaled = np.concatenate(
                    (
                        self.test_scaled,
                        self.dtf_test[["hour", "day", "weekday", "month"]].values,
                    ),
                    axis=1,
                )
                self.test_small_scaled = np.concatenate(
                    (
                        self.test_small_scaled,
                        self.dtf_test_small[["hour", "day", "weekday", "month"]].values,
                    ),
                    axis=1,
                )

            self.scaler = scaler

            def to_ds(ds, step, raw_data):
                out_features_mask = None

                ds = ContextBatchDataset(
                    Features=torch.from_numpy(ds[:, self.bahnlast_mask]).to(
                        torch.float32
                    ),
                    Context=torch.from_numpy(ds[:, ~self.bahnlast_mask]).to(
                        torch.float32
                    ),
                    out_features_mask=out_features_mask,
                    ts_in=self.ts_in,
                    ts_out=self.ts_out,
                    step=step,
                    raw_data=raw_data,
                )
                return ds

            self.datasets = dict(
                train_test=to_ds(self.train_scaled, self.step_test, self.train),
                train=to_ds(self.train_scaled, self.step_train, self.train),
                val=to_ds(self.val_scaled, self.step_val, self.val),
                test=to_ds(self.test_scaled, self.step_test, self.test),
                test_small=to_ds(self.test_small_scaled, self.step_test, self.test),
            )

    def inverse_scale_bahnlast(self, X):
        x = np.zeros((X.shape[0], self.train_scaled.shape[1]))
        x[:, self.bahnlast_mask] = X

        if self.datetime_features:
            res = self.scaler.inverse_transform(x[:, self.scaler_mask])
        else:
            res = self.scaler.inverse_transform(x)

        return res[:, self.bahnlast_mask_orig]

    def _create_dataloader(self, ds, batch_size, shuffle=True):
        sampler = BatchSampler(
            RandomSampler(ds) if shuffle else SequentialSampler(ds),
            batch_size=batch_size,
            drop_last=self.drop_last,
        )
        loader_args = dict(
            sampler=sampler,
            num_workers=self.num_workers,
            batch_size=None,  # otherwise it adds a dimension with size 1 x ...
        )
        return DataLoader(ds, **loader_args)

    def train_dataloader(self):
        return self._create_dataloader(
            self.datasets["train"], batch_size=self.batch_size
        )

    def val_dataloader(self):
        return self._create_dataloader(
            self.datasets["val"], batch_size=self.batch_size, shuffle=False
        )

    def test_dataloader(self):
        return self._create_dataloader(
            self.datasets["test"], batch_size=self.batch_size, shuffle=False
        )

    def test_small_dataloader(self):
        return self._create_dataloader(
            self.datasets["test_small"], batch_size=self.batch_size, shuffle=False
        )

# Regression Benchmark Railway

In [9]:
data = pd.read_csv("../datasets/railway.csv", sep=",")
data.datetime = pd.to_datetime(data.datetime)
data.set_index("datetime", inplace=True)

In [10]:
# Every day of has 24 observations corresponding to each hour.
data.iloc[20:28]

,Locarno / Monti,Pully,Wynau,Zürich / Fluntern,Bahnlast,Tonnage_F_Z,Tonnage_R_Z,Tonnage_G_Z,Kilometerleistung_F_Z,Kilometerleistung_R_Z,...,Tonnage_G_W,Kilometerleistung_F_W,Kilometerleistung_R_W,Kilometerleistung_G_W,Anzahl_F_W,Anzahl_R_W,Anzahl_G_W,Btkm_F_W,Btkm_R_W,Btkm_G_W
datetime,,,,,,,,,,,,,,,,,,,,,
2018-04-02 20:00:00,11.425,9.200,8.725,10.800,240.286225,231386.945,159977.905,61656.145,885.571949,1411.443299,...,18268.075,320.377365,818.110577,52.386991,14.521667,51.353333,4.098333,178063.497400,166654.004805,35833.577669
2018-04-02 21:00:00,11.400,9.175,7.725,10.150,234.067475,253317.025,131155.760,35507.155,945.815783,1164.527110,...,15547.225,300.333417,820.430911,58.997815,13.441667,52.161667,4.200000,171539.455553,155072.959105,30135.847762
2018-04-02 22:00:00,11.075,8.850,6.850,9.750,197.381300,155176.615,106939.580,67243.300,628.368266,1008.493993,...,23535.750,310.955778,875.503302,90.466796,13.456667,55.563333,5.540000,184197.588741,165501.169041,46501.514998
2018-04-02 23:00:00,10.625,9.675,6.700,9.425,173.647725,95774.200,102667.885,53448.600,373.148332,968.085488,...,19955.050,245.894352,750.854187,82.079148,10.585000,47.626667,4.403333,163781.693397,152419.184503,57052.144223
2018-04-03 00:00:00,10.575,8.875,7.100,9.325,121.712525,64261.775,70171.165,70140.530,258.044554,561.208299,...,34529.375,111.408166,552.450849,71.022773,5.716667,37.035000,5.598333,76341.609925,113581.095197,67803.175162
2018-04-03 01:00:00,10.125,9.700,7.475,9.675,68.501475,24005.900,8447.905,77009.930,98.851106,83.667406,...,25540.500,29.033031,116.878590,43.205809,1.178333,7.295000,3.085000,17242.695434,23529.590031,48510.202716
2018-04-03 02:00:00,10.250,8.650,6.900,9.250,61.683250,0.000,0.000,89962.875,0.000000,0.000000,...,23055.250,0.000000,0.000000,34.003538,0.000000,0.000000,2.783333,0.000000,0.000000,54430.470031
2018-04-03 03:00:00,10.650,9.425,6.475,9.175,71.856750,0.000,0.000,106647.000,0.000000,0.000000,...,53327.750,0.000000,0.000000,76.497577,0.000000,0.000000,5.181667,0.000000,0.000000,113282.957074


In [11]:
dm = SpatialDataModule(ts_in=24, batch_size=1)
dm.setup()

In [12]:
def dl_to_sklearn_with_context(dl):
    X, y = [], []
    for batch in iter(dl):
        x_c, y_c, x_t, y_t = batch
        X.append(
            torch.concat(
                [torch.concat([y_c, x_c], dim=2).flatten(1), x_t.flatten(1)], dim=1
            )
        )
        y.append(y_t.flatten(1))

    return torch.concat(X).numpy(), torch.concat(y).numpy()


def dl_to_sklearn__with_context_for_tabpfn(dl, lag=0):
    X, y = [], []
    for batch in iter(dl):
        # x_c.shape is (batch_size, ts_in, n_features), ts_in is 24 for railway datset (I guess that 24h observations, no?)
        x_c, y_c, x_t, y_t = batch
        X.append(torch.concat([x_t[:, lag, :], y_c.squeeze(2)], dim=1))
        y.append(y_t[:, lag, :])

    return torch.concat(X).numpy(), torch.concat(y).numpy()


def dl_to_context_for_tabpfn(dl, lag=0, device="cuda:0"):
    X, y = [], []
    for batch in iter(dl):
        # x_c.shape is (batch_size, ts_in, n_features), ts_in is 24 for railway datset (I guess that 24h observations, no?)
        x_c, y_c, x_t, y_t = batch
        X.append(torch.concat([x_t[:, lag, :], y_c.squeeze(2)], dim=1))
        y.append(y_t[:, lag, :])

    return torch.concat(X).to(device), torch.concat(y).to(device)


def dl_to_sklearn_no_context(dl):
    X, y = [], []
    for batch in iter(dl):
        x_c, y_c, x_t, y_t = batch
        X.append(torch.concat([y_c, x_c], dim=2).flatten(1))
        y.append(y_t.flatten(1))

    return torch.concat(X).numpy(), torch.concat(y).numpy()

In [13]:
batch = next(iter(dm.train_dataloader()))
x_c, y_c, x_t, y_t = batch

In [14]:
x_t.shape, y_t.shape, x_c.shape, y_c.shape

(torch.Size([1, 24, 56]),
 torch.Size([1, 24, 1]),
 torch.Size([1, 24, 56]),
 torch.Size([1, 24, 1]))

In [15]:
for lag in [0]:
    X_train, y_train = dl_to_sklearn__with_context_for_tabpfn(
        dm.train_dataloader(), lag=lag
    )
    print("Lag: ", lag)
    print(f"X_train.shape: {X_train.shape}, y_train.shape: {y_train.shape}")

Lag:  0
X_train.shape: (39193, 80), y_train.shape: (39193, 1)
Lag:  1
X_train.shape: (39193, 80), y_train.shape: (39193, 1)
Lag:  2
X_train.shape: (39193, 80), y_train.shape: (39193, 1)


In [16]:
torch.concat([x_t[:, 0, :], y_c.squeeze(2)], dim=1).shape, y_c.squeeze(2)[:, 0].shape

(torch.Size([1, 80]), torch.Size([1]))

This appliaction of TabPFN is not valid as TabPFN alternated between feature attention and item attention, hence whenever we pass test sets with that have some future information we leak the future into the predictions.

In [120]:
data = pd.read_csv("../datasets/railway.csv", sep=",")
data.datetime = pd.to_datetime(data.datetime)
data.set_index("datetime", inplace=True)
data.index.set_names("timestamp", inplace=True)
data.rename(columns={"Bahnlast": "target"}, inplace=True)


# add additional index id, the id of the day hence every day has 24 observations
day_ids = data.index.date

# convert day_ids to to a range of integers, assigning each day a unique id
day_ids = pd.Series(day_ids).astype("category").cat.codes.astype("int")
data["item_id"] = day_ids.values
data = data[["target"]]
data.item_ids = data.index.get_level_values("item_id").unique()

KeyError: 'Requested level (item_id) does not match index name (timestamp)'

In [ ]:
from tabpfn_time_series.data_preparation import generate_test_X

prediction_length = 8208
train_tsdf, test_tsdf_ground_truth = data.train_test_split(
    prediction_length=prediction_length
)
test_tsdf = generate_test_X(train_tsdf, prediction_length)

In [ ]:
# make the double index where day_ids is he first index and current index is the second
data = data.set_index(["item_id", data.index])

In [110]:
data = data[["target"]]
data.item_ids = data.index.get_level_values("item_id").unique()

/tmp/ipykernel_1193457/170391943.py:2: UserWarning: Pandas doesn't allow columns to be created via a new attribute name - see https://pandas.pydata.org/pandas-docs/stable/indexing.html#attribute-access
  data.item_ids = data.index.get_level_values("item_id").unique()


In [112]:
# take last 8208 samplples as test
data_test = data.iloc[-8208:, :].copy()
data_test["target"] = np.nan

In [113]:
from tabpfn_time_series import FeatureTransformer, DefaultFeatures

selected_features = [
    DefaultFeatures.add_running_index,
    DefaultFeatures.add_calendar_features,
]

train_tsdf, test_tsdf = FeatureTransformer.add_features(
    data,
    data_test,
    selected_features,
)

AssertionError: 

In [116]:
assert data_test["target"].isna().all(), "bla"

In [101]:
from tabpfn_time_series import TabPFNTimeSeriesPredictor, TabPFNMode

predictor = TabPFNTimeSeriesPredictor(tabpfn_mode=TabPFNMode.LOCAL)

pred = predictor.predict(
    data,
    data_test,
)

KeyError: '[2085 1941 1727 1170  723 1323 1055  905 1570 1825 1191 1317  530  135\n  385 1520  264 1475  145 1928  489 2054 1192 1648  789 1629  744 2089\n  438 1780 1988  621 1175  862  333 1873   48 2033   76  453 1511 2045\n  648   70 1505 1859  251 1585  863 1349  253  475  728   18 1820 1584\n 1340 1367 1108 1335 1726  793  564 1339 1042  817] not in index'

In [84]:
data_test

Locarno / Monti  Pully  Wynau  \
item_ids timestamp                                            
1758     2023-01-24 00:00:00            4.525  0.050  1.400   
         2023-01-24 01:00:00            4.150 -0.425  1.300   
         2023-01-24 02:00:00            4.150 -0.475  1.250   
         2023-01-24 03:00:00            3.800 -0.225  1.300   
         2023-01-24 04:00:00            3.550 -0.500  1.200   
...                                       ...    ...    ...   
2099     2023-12-31 19:00:00            6.975  5.775  1.925   
         2023-12-31 20:00:00            6.800  5.725  2.125   
         2023-12-31 21:00:00            6.300  5.975  2.200   
         2023-12-31 22:00:00            5.500  5.800  2.375   
         2023-12-31 23:00:00            4.900  5.950  3.175   

                              Zürich / Fluntern  target   Tonnage_F_Z  \
item_ids timestamp                                                      
1758     2023-01-24 00:00:00              0.825     NaN   83616.77305   
         2023-01-24 01:00:00              0.850     NaN   16056.91310   
         2023-01-24 02:00:00              0.975     NaN     223.25000   
         2023-01-24 03:00:00              0.775     NaN       0.00000   
         2023-01-24 04:00:00              0.575     NaN       0.00000   
...                                         ...     ...           ...   
2099     2023-12-31 19:00:00              3.900     NaN  322295.32025   
         2023-12-31 20:00:00              3.700     NaN  303708.22750   
         2023-12-31 21:00:00              3.675     NaN  228431.98745   
         2023-12-31 22:00:00              3.375     NaN  123301.23100   
         2023-12-31 23:00:00              3.700     NaN   80580.35475   

                              Tonnage_R_Z   Tonnage_G_Z  \
item_ids timestamp                                        
1758     2023-01-24 00:00:00   99475.5490  167429.87170   
         2023-01-24 01:00:00   12779.1020  254013.16575   
         2023-01-24 02:00:00      54.5000  272025.00000   
         2023-01-24 03:00:00       0.0000  315308.25000   
         2023-01-24 04:00:00    3508.7500  294421.73650   
...                                   ...           ...   
2099     2023-12-31 19:00:00  193752.2546   32028.03965   
         2023-12-31 20:00:00  199940.0428   31077.38590   
         2023-12-31 21:00:00  162247.2166   45758.58450   
         2023-12-31 22:00:00  139179.5681   32643.57920   
         2023-12-31 23:00:00  152772.5322   24757.06375   

                              Kilometerleistung_F_Z  Kilometerleistung_R_Z  \
item_ids timestamp                                                           
1758     2023-01-24 00:00:00             313.803879             725.865882   
         2023-01-24 01:00:00              76.140464             138.685646   
         2023-01-24 02:00:00               0.000000               0.187000   
         2023-01-24 03:00:00               0.000000               0.000000   
         2023-01-24 04:00:00               0.000000              35.612241   
...                                             ...                    ...   
2099     2023-12-31 19:00:00            1015.787367            1388.582194   
         2023-12-31 20:00:00            1012.424801            1453.518218   
         2023-12-31 21:00:00             845.101115            1217.553294   
         2023-12-31 22:00:00             467.818903            1096.397096   
         2023-12-31 23:00:00             270.949889            1117.379149   

                              ...  Tonnage_G_W  Kilometerleistung_F_W  \
item_ids timestamp            ...                                       
1758     2023-01-24 00:00:00  ...  150021.0967             125.717886   
         2023-01-24 01:00:00  ...  180786.2500              44.887945   
         2023-01-24 02:00:00  ...  221671.7500               0.000000   
         2023-01-24 03:00:00  ...  249415.7500               0.000000   
         2023-01-24 04:00:00  ...  2872

In [69]:
data_test["target"].isna().all()

True

In [82]:
from sklearn.metrics import mean_squared_error, r2_score

from tqdm import tqdm

# Assuming there is a TabPFNRegressor (if not, a different regressor should be used)
from tabpfn import TabPFNRegressor

feature_mode = "exogen"  # "endogen" or "exogen"
model_name = "tabpfn"
seed = -1
results = []
N_samples_train = 100
N_samples_test = 1000
factor_agg = pd.read_csv("../datasets/railway_agg.csv")["Bahnlast"].abs().mean()
factor_full = pd.read_csv("../datasets/railway.csv")["Bahnlast"].abs().mean()
# factor_full = 1


pbar = tqdm(range(24))  #
for lag in pbar:
    if feature_mode == "endogen":
        X_train, y_train = dl_to_sklearn_no_context(dm.train_dataloader())
        X_test, y_test = dl_to_sklearn_no_context(dm.test_dataloader())
    elif feature_mode == "exogen":
        X_train, y_train = dl_to_sklearn__with_context_for_tabpfn(
            dm.train_dataloader(), lag=lag
        )
        X_test, y_test = dl_to_sklearn__with_context_for_tabpfn(
            dm.test_dataloader(), lag=lag
        )
    else:
        raise ValueError("Feature mode unknown")

    if N_samples_train is not None:
        X_train, y_train = X_train[-N_samples_train:], y_train[-N_samples_train:]

    if N_samples_test is not None:
        X_test, y_test = X_test[:N_samples_test], y_test[:N_samples_test]

    # Initialize the regressor
    regressor = TabPFNRegressor(
        ignore_pretraining_limits=True, device="cuda:0"
    )  # AutoTabPFNRegressor(max_time=10, device="cuda:0") #

    regressor.fit(X_train, y_train)

    # Predict on the test set

    # To make sure that we actually not leaking the data we have to go over the test set one by one.
    # y_test_hat = regressor.predict(X_test)
    y_test_hat = []
    for i in range(len(X_test)):
        y_test_hat.append(regressor.predict(X_test[i].reshape(1, -1)))

    y_test_hat = np.array(y_test_hat).flatten()

    # Evaluate the model
    mse = mean_squared_error(y_test, y_test_hat)
    r2 = r2_score(y_test, y_test_hat)

    # print("Mean Squared Error (MSE):", mse)
    # print("R² Score:", r2)

    pbar.set_postfix(mse=f"{mse:.4f}", R2=f"R² Score: {r2:.4f}")

    y_test_hat_rs = dm.inverse_scale_bahnlast(rearrange(y_test_hat, "B -> (B) 1"))
    y_test_rs = dm.inverse_scale_bahnlast(rearrange(y_test, "B F -> (B F) 1"))

    results.append(
        {
            "exp_name": f"{model_name}_{feature_mode}",
            "seed": seed,
            "lag": lag,
            "normalized/test/mae": np.abs((y_test_hat_rs - y_test_rs)).mean()
            * 100
            / factor_full,
            "normalized/test/mse": np.abs((y_test_hat_rs - y_test_rs) ** 2).mean()
            * 100
            / factor_full,
        }
    )

df = pd.DataFrame(results).set_index(["exp_name", "seed"])
df
# df.to_csv(f"railway/{model_name}_{feature_mode}.csv")

 21%|██        | 5/24 [12:03<45:47, 144.63s/it, R2=R² Score: 0.8000, mse=0.0016]


KeyboardInterrupt: 

In [83]:
df = pd.DataFrame(results).set_index(["exp_name", "seed"])
df

lag  normalized/test/mae  normalized/test/mse
exp_name      seed                                               
tabpfn_exogen -1      0             5.257305           110.521410
              -1      1             5.184073           102.598496
              -1      2             3.248145            46.585592
              -1      3             4.450231            84.404192
              -1      4             5.737835           127.430572

In [84]:
df.mean()

lag                     2.000000
normalized/test/mae     4.775518
normalized/test/mse    94.308052
dtype: float64

In [12]:
df

lag  normalized/test/mae  normalized/test/mse
exp_name      seed                                               
tabpfn_exogen -1      0             2.823262            32.436024
              -1      1             3.242528            45.473954
              -1      2             2.824748            35.730222
              -1      3             3.506425            51.621711
              -1      4             2.976210            36.767955
              -1      5             4.158617            76.044689
              -1      6             4.927968            97.956360
              -1      7             3.671031            58.807399
              -1      8             3.577567            51.351199
              -1      9             4.126672            71.357725
              -1     10             3.674114            60.052820
              -1     11             3.729249            57.980918
              -1     12             4.034656            67.636591
              -1     13             3.439914            50.361704
              -1     14             3.899217            63.754154
              -1     15             3.812291            62.793984
              -1     16             4.907510            97.532994
              -1     17             3.759366            63.155839
              -1     18             4.284598            76.421982
              -1     19             3.924257            64.659411
              -1     20             4.130199            69.585913
              -1     21             3.367872            48.991666
              -1     22             3.823603            57.592994
              -1     23             3.263017            45.198708

In [13]:
# from xgboost import XGBRegressor
# from catboost import CatBoostRegressor
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
import os
import random
from joblib import dump

seeds = [
    72,
]  #  88, 101, 666, 226688

# Make sure results folder exists
results_folder = "../results/railway"
os.makedirs(results_folder, exist_ok=True)

for model_name in ["LinearRegression", "KNN"]:  # "MLP"
    for feature_mode in ["endogen", "exogen"]:
        results = []
        for seed in seeds:
            random.seed(seed)
            os.environ["PYTHONHASHSEED"] = str(seed)
            np.random.seed(seed)
            torch.manual_seed(seed)
            torch.cuda.manual_seed_all(seed)

            if feature_mode == "endogen":
                X_train, y_train = dl_to_sklearn_no_context(dm.train_dataloader())
                X_test, y_test = dl_to_sklearn_no_context(dm.test_dataloader())
            elif feature_mode == "exogen":
                X_train, y_train = dl_to_sklearn_with_context(dm.train_dataloader())
                X_test, y_test = dl_to_sklearn_with_context(dm.test_dataloader())
            else:
                raise ValuerError("Feature mode unknown")

            if model_name == "Catboost":
                pass
                # model = CatBoostRegressor(iterations=1000, learning_rate=0.03, depth=4, silent=True, loss_function='MultiRMSE', random_seed=seed)
            elif model_name == "XGBoost":
                pass
                # model = XGBRegressor(verbosity=0, objective='reg:squarederror', n_estimators=1000, device="cuda", random_state=seed)
            elif model_name == "LinearRegression":
                model = LinearRegression()
            elif model_name == "KNN":
                model = KNeighborsRegressor()
            elif model_name == "MLP":
                model = MLPRegressor(
                    hidden_layer_sizes=(100, 100),
                    batch_size=64,
                    max_iter=1000,
                    early_stopping=True,
                    validation_fraction=0.2,
                    random_state=seed,
                )
            elif model_name == "TabPFN":
                model = TabPFNRegressor()
            else:
                raise ValuerError("Model name unknown")

            model.fit(X_train, y_train)

            print("Evaluating model with seed", seed)

            y_test_hat = model.predict(X_test)

            np.save(
                f"{results_folder}/{model_name}_{feature_mode}_{seed}_forecasts.npy",
                y_test_hat,
            )
            if model_name == "Catboost":
                model.save_model(
                    f"{results_folder}/Catboost_{feature_mode}_{seed}_model.json"
                )
            elif model_name == "XGBoost":
                model.save_model(
                    f"{results_folder}/XGBoost_{feature_mode}_{seed}_model.json"
                )
            elif model_name == "LinearRegression":
                dump(
                    model,
                    f"{results_folder}/LinearRegression_{feature_mode}_{seed}_model.joblib",
                )
            elif model_name == "KNN":
                dump(model, f"{results_folder}/KNN_{feature_mode}_{seed}_model.joblib")
            elif model_name == "MLP":
                dump(model, f"{results_folder}/MLP_{feature_mode}_{seed}_model.joblib")
            else:
                raise ValuerError("Model name unknown")

            y_test_hat_rs = dm.inverse_scale_bahnlast(
                rearrange(y_test_hat, "B T -> (B T) 1")
            )
            y_test_rs = dm.inverse_scale_bahnlast(rearrange(y_test, "B T -> (B T) 1"))

            results.append(
                {
                    "exp_name": f"{model_name}_{feature_mode}",
                    "seed": seed,
                    "normalized/test/mae": np.abs((y_test_hat_rs - y_test_rs)).mean(),
                    "normalized/test/mse": np.abs(
                        (y_test_hat_rs - y_test_rs) ** 2
                    ).mean(),
                }
            )

        df = pd.DataFrame(results).set_index(["exp_name", "seed"])

        df.to_csv(f"{results_folder}/{model_name}_{feature_mode}.csv")

Evaluating model with seed 72
Evaluating model with seed 72
Evaluating model with seed 72


OpenBLAS warning: precompiled NUM_THREADS exceeded, adding auxiliary array for thread metadata.
To avoid this warning, please rebuild your copy of OpenBLAS with a larger NUM_THREADS setting
or set the environment variable OPENBLAS_NUM_THREADS to 64 or lower


Evaluating model with seed 72


### Persistence

In [16]:
# PERSISTENCE
forecast_horizon = 24
seasonal_lag = 168
stride = 24

# Generate test data
dm = SpatialDataModule()
dm.setup()

_, _ = dl_to_sklearn_no_context(dm.train_dataloader())
_, y_test = dl_to_sklearn_no_context(dm.test_dataloader())

# Load target data
target_data = pd.read_csv("../datasets/railway.csv").set_index("datetime")["Bahnlast"]
target_data.index = pd.to_datetime(target_data.index)

# Vectorized extraction of the lagged forecast matrix
persistence_base = target_data.shift(seasonal_lag)
persistence_matrix = pd.concat(
    [persistence_base.shift(-i) for i in range(forecast_horizon)], axis=1
).dropna()

# Rearrange
y_test_hat_rs = rearrange(
    persistence_matrix[::stride]
    .loc["2023-01-24 00:00:00":][int(seasonal_lag / forecast_horizon) :]
    .values,
    "B T -> (B T) 1",
)
y_test_rs = dm.inverse_scale_bahnlast(rearrange(y_test, "B T -> (B T) 1"))

for feature_mode in ["endogen", "exogen"]:
    benchmark = []
    for seed in seeds:
        np.save(
            f"{results_folder}/WeeklyNaive_{feature_mode}_{seed}_forecasts.npy",
            persistence_matrix[::stride]
            .loc["2023-01-24 00:00:00":][int(seasonal_lag / forecast_horizon) :]
            .values,
        )
        benchmark.append(
            {
                "exp_name": f"WeeklyNaive_{feature_mode}",
                "seed": seed,
                "normalized/test/mae": np.abs((y_test_hat_rs - y_test_rs)).mean(),
                "normalized/test/mse": np.abs((y_test_hat_rs - y_test_rs) ** 2).mean(),
            }
        )
    df = pd.DataFrame(benchmark).set_index(["exp_name", "seed"])
    df.to_csv(f"{results_folder}/WeeklyNaive_{feature_mode}.csv")

# Regression Benchmarks Buildings

In [18]:
dm = BuildingEnergyDataModule(
    data_dir="../datasets/alpha_building_years_02_03_occupancy_levelwise.csv",
    test_ratio=0.5,
)
dm.load_raw_data(
    data_dir="../datasets/alpha_building_years_02_03_occupancy_levelwise.csv"
)
dm.setup()

AssertionError: Should be boolean vector in order 1111...000

In [ ]:
def dl_to_sklearn_with_context(dl):
    X, y = [], []
    for batch in iter(dl):
        x_c, y_c, x_t, y_t = batch
        X.append(
            torch.concat(
                [torch.concat([y_c, x_c], dim=2).flatten(1), x_t.flatten(1)], dim=1
            )
        )
        y.append(y_t.flatten(1))

    return torch.concat(X).numpy(), torch.concat(y).numpy()


def dl_to_sklearn_no_context(dl):
    X, y = [], []
    for i, batch in enumerate(iter(dl)):
        x_c, y_c, x_t, y_t = batch
        X.append(torch.concat([y_c, x_c], dim=2).flatten(1))
        y.append(y_t.flatten(1))

    return torch.concat(X).numpy(), torch.concat(y).numpy()

In [ ]:
# from xgboost import XGBRegressor
# from catboost import CatBoostRegressor
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.multioutput import MultiOutputRegressor

# import lightgbm as lgb
import os
import random
from joblib import dump

seeds = [72, 88, 101, 666, 226688]

for model_name in [
    "LinearRegression",
    "KNN",
    "MLP",
]:  # "XGBoost", "Catboost"
    for feature_mode in ["endogen", "exogen"]:
        # Train and evaluate models with different random seeds
        results = []
        for seed in seeds:
            random.seed(seed)
            os.environ["PYTHONHASHSEED"] = str(seed)
            np.random.seed(seed)
            torch.manual_seed(seed)
            torch.cuda.manual_seed_all(seed)

            if feature_mode == "endogen":
                X_train, y_train = dl_to_sklearn_no_context(dm.train_dataloader())
                X_test, y_test = dl_to_sklearn_no_context(dm.test_dataloader())
            elif feature_mode == "exogen":
                X_train, y_train = dl_to_sklearn_with_context(dm.train_dataloader())
                X_test, y_test = dl_to_sklearn_with_context(dm.test_dataloader())
            else:
                raise ValuerError("Feature mode unknown")

            if model_name == "Catboost":
                model = CatBoostRegressor(
                    iterations=1000,
                    learning_rate=0.03,
                    depth=4,
                    silent=True,
                    loss_function="MultiRMSE",
                    random_seed=seed,
                )
            elif model_name == "XGBoost":
                model = XGBRegressor(
                    verbosity=0,
                    objective="reg:squarederror",
                    n_estimators=1000,
                    device="cuda",
                    random_state=seed,
                )
            elif model_name == "LightGBM":
                model = model = MultiOutputRegressor(
                    lgb.LGBMRegressor(
                        boosting_type="gbdt", n_estimators=1000, verbose=-1
                    )
                )
            elif model_name == "LinearRegression":
                model = LinearRegression()
            elif model_name == "KNN":
                model = KNeighborsRegressor()
            elif model_name == "MLP":
                model = MLPRegressor(
                    hidden_layer_sizes=(100, 100),
                    batch_size=64,
                    max_iter=1000,
                    early_stopping=True,
                    validation_fraction=0.2,
                    random_state=seed,
                )
            else:
                raise ValuerError("Model name unknown")

            model.fit(X_train, y_train)

            print("Evaluating model with seed", seed)

            y_test_hat = model.predict(X_test)

            np.save(
                f"building/{model_name}_{feature_mode}_{seed}_forecasts.npy", y_test_hat
            )
            if model_name == "Catboost":
                model.save_model(f"building/Catboost_{feature_mode}_{seed}_model.json")
            elif model_name == "XGBoost":
                model.save_model(f"building/XGBoost_{feature_mode}_{seed}_model.json")
            elif model_name == "LightGBM":
                dump(model, f"building/LightGBM_{feature_mode}_{seed}_model.joblib")
            elif model_name == "LinearRegression":
                dump(
                    model,
                    f"building/LinearRegression_{feature_mode}_{seed}_model.joblib",
                )
            elif model_name == "KNN":
                dump(model, f"building/KNN_{feature_mode}_{seed}_model.joblib")
            elif model_name == "MLP":
                dump(model, f"building/MLP_{feature_mode}_{seed}_model.joblib")
            else:
                raise ValuerError("Model name unknown")

            y_test_hat_rs = rearrange(y_test_hat, "B T -> (B T) 1")
            y_test_rs = rearrange(y_test, "B T -> (B T) 1")

            results.append(
                {
                    "exp_name": f"{model_name}_{feature_mode}",
                    "seed": seed,
                    "normalized/test/mae": np.abs((y_test_hat_rs - y_test_rs)).mean(),
                    "normalized/test/mse": np.abs(
                        (y_test_hat_rs - y_test_rs) ** 2
                    ).mean(),
                }
            )

        df = pd.DataFrame(results).set_index(["exp_name", "seed"])
        df.to_csv(f"building/{model_name}_{feature_mode}.csv")

Evaluating model with seed 72


FileNotFoundError: [Errno 2] No such file or directory: 'building/LinearRegression_endogen_72_forecasts.npy'

/home/levtel/miniconda3/envs/phm/bin/python


### Persistence

In [ ]:

# PERSISTENCE
seeds = [72, 88, 101, 666, 226688]
forecast_horizon = 48
seasonal_lag = 336
stride = 48

# Generate test data
dm = BuildingEnergyDataModule(
    data_dir="C:/Users/vole/Downloads/Building_Operation_Data/alpha_building_years_02_03_occupancy_levelwise.csv",
    test_ratio=0.5,
)
dm.load_raw_data(
    data_dir="C:/Users/vole/Downloads/Building_Operation_Data/alpha_building_years_02_03_occupancy_levelwise.csv"
)
dm.setup()

_, _ = dl_to_sklearn_no_context(dm.train_dataloader())
_, y_test = dl_to_sklearn_no_context(dm.test_dataloader())

# Load target data
target_data = pd.read_csv(
    "C:/Users/vole/Downloads/Building_Operation_Data/alpha_building_years_02_03_occupancy_levelwise.csv"
).set_index("Unnamed: 0")["Site Total Energy (kWh)"]
target_data.index = pd.to_datetime(target_data.index)


# Scale
scaler = MinMaxScaler()
scaler.fit(target_data.loc["2002"].values.reshape(-1, 1))
scaled_target_data = pd.Series(
    scaler.transform(target_data.values.reshape(-1, 1)).squeeze(),
    index=target_data.index,
)

# Vectorized extraction of the lagged forecast matrix
persistence_base = scaled_target_data.shift(seasonal_lag)
persistence_matrix = pd.concat(
    [persistence_base.shift(-i) for i in range(forecast_horizon)], axis=1
).dropna()

# Rearrange
y_test_hat_rs = rearrange(
    persistence_matrix[::stride]
    .loc["2003-01-01":][int(seasonal_lag / forecast_horizon) :]
    .values,
    "B T -> (B T) 1",
)
y_test_rs = rearrange(y_test, "B T -> (B T) 1")

for feature_mode in ["endogen", "exogen"]:
    benchmark = []
    for seed in seeds:
        np.save(
            f"building/WeeklyNaive_{feature_mode}_{seed}_forecasts.npy",
            persistence_matrix[::stride]
            .loc["2003-01-01":][int(seasonal_lag / forecast_horizon) :]
            .values,
        )
        benchmark.append(
            {
                "exp_name": f"WeeklyNaive_{feature_mode}",
                "seed": seed,
                "normalized/test/mae": np.abs((y_test_hat_rs - y_test_rs)).mean(),
                "normalized/test/mse": np.abs((y_test_hat_rs - y_test_rs) ** 2).mean(),
            }
        )
    df = pd.DataFrame(benchmark).set_index(["exp_name", "seed"])
    df.to_csv(f"building/WeeklyNaive_{feature_mode}.csv")

# ARIMA

In [ ]:
import statsmodels.api as sm
from statsmodels.tsa.seasonal import MSTL
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (15, 10)

target_data = pd.read_csv(
    "C:/Users/vole/Downloads/Building_Operation_Data/alpha_building_years_02_03_occupancy_levelwise.csv"
).set_index("Unnamed: 0")["Site Total Energy (kWh)"]
target_data.index = pd.to_datetime(target_data.index)

# sm.graphics.tsa.plot_acf(target_data.values.squeeze(), lags=340)

res = MSTL(target_data.loc["2002"], periods=(48, 48 * 7)).fit()
res.plot()
residuals = res.resid
print("3")

In [ ]:
# Plot PACF for lag order
sm.graphics.tsa.plot_pacf(residual.diff().iloc[2:], lags=336)
plt.show()

In [ ]:
sm.graphics.tsa.plot_acf(residual.diff().iloc[2:], lags=336)
plt.show()

In [ ]:
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA
from statsforecast.arima import arima_string

df = target_data.loc["2002"].reset_index()
df["unique id"] = "1"
df.columns = ["ds", "y", "unique_id"]
df["ds"] = pd.to_datetime(df["ds"])

sf = StatsForecast(
    models=[AutoARIMA(max_p=336, max_d=3, max_q=336, max_P=336, max_D=3, max_Q=336)],
    freq="30min",
)
sf.fit(df=df)
arima_string(sf.fitted_[0, 0].model_)

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf
import seaborn as sns
import scipy.stats as stats

result = sf.fitted_[0, 0].model_
print(result.keys())
print(result["arma"])

residual = pd.DataFrame(result.get("residuals"), columns=["residual Model"])
residual

fig, axs = plt.subplots(nrows=2, ncols=2)

# plot[1,1]
residual.plot(ax=axs[0, 0])
axs[0, 0].set_title("Residuals")

# plot
sns.distplot(residual, ax=axs[0, 1])
axs[0, 1].set_title("Density plot - Residual")

# plot
stats.probplot(residual["residual Model"], dist="norm", plot=axs[1, 0])
axs[1, 0].set_title("Plot Q-Q")

# plot
plot_acf(residual, lags=336, ax=axs[1, 1], color="fuchsia")
axs[1, 1].set_title("Autocorrelation")

plt.show()

In [ ]:
target_data.loc["2002"]

# Collect results

In [ ]:
results = []
for name in ["XGBoost", "Catboost", "LinearRegression", "KNN", "MLP", "WeeklyNaive"]:
    for mode in ["endogen", "exogen"]:
        results.append(pd.read_csv(f"building/{name}_{mode}.csv"))
pd.concat(results).to_csv("all_benchmarks_building.csv")

In [ ]:
results = []
for name in ["LinearRegression", "KNN", "MLP", "WeeklyNaive"]:
    for mode in ["endogen", "exogen"]:
        results.append(pd.read_csv(f"railway/{name}_{mode}.csv"))
pd.concat(results).to_csv("all_benchmarks_railway.csv")